## LAB03 · Ensambles, reducción dimensional y Green AI

**Pregunta guía:** manteniendo exactamente el mismo dataset, target y partición del Ejercicio 01 (LAB02, fuga de clientes), ¿qué tan lejos me lleva ampliar el catálogo de modelos a ensambles, y qué me cuesta en tiempo de cómputo cada punto extra de F1 que gano?

Este notebook **no cambia** el dataset, el `TARGET`, las columnas retiradas ni la partición train/test de `notebooks/02_churn_svm.ipynb` — los reconstruyo aquí de forma idéntica (mismo `random_state=42`, mismo `test_size=.20`, mismo `preprocess`) para poder comparar seis configuraciones de modelo sobre el mismo terreno, en vez de reentrenar sobre un split distinto y confundir "mejoró el modelo" con "cambió la partición".

### Paso 1 — Congelar y confirmar el protocolo de LAB02

Repito acá, sin modificar nada, el bloque de descarga + target + preprocesamiento del notebook anterior. La única diferencia de forma es que uso `download_iranian_churn_dataset()` otra vez (ya cacheado en `data/raw/dataset.csv` si ya lo corrí antes), y vuelvo a armar `preprocess`, `X`, `y`, `Xtr/Xte/ytr/yte` exactamente igual que en `02_churn_svm.ipynb` — mismo `TARGET`, mismo `DROP_COLUMNS`, mismo `random_state`. Si algún día cambio algo de esto sin querer, los tests de `tests/test_data_contract.py` (que corren sobre el mismo dataset) me avisarían.

In [ ]:
import sys
from pathlib import Path

# Mismo problema de siempre en este proyecto: el kernel de Jupyter corre
# con cwd = notebooks/, no la raíz del repo. Ver notebooks/02_churn_svm.ipynb
# para la explicación completa; acá repito la solución y agrego una
# segunda entrada a sys.path: la raíz del proyecto (para "from src....",
# igual que en LAB02) y también "src/" (para "from inf8239_u01...", que
# es como lo importan tests/test_green.py y el mandato del LAB03 -- los
# tests corren con PYTHONPATH=src, así que replico eso mismo acá en vez
# de tener dos formas distintas de resolver el mismo paquete).
project_root = Path.cwd()
if not (project_root / "requirements.txt").exists() and (project_root.parent / "requirements.txt").exists():
    project_root = project_root.parent
for candidate in (project_root, project_root / "src"):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from src.inf8239_u01.data import download_iranian_churn_dataset
path = download_iranian_churn_dataset()
print(path)

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv(path)

TARGET = "Churn"
DROP_COLUMNS = ["Age"]  # idéntico a LAB02: no documentada por UCI, duplica Age Group
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
# handle_unknown="ignore" ya estaba en LAB02; agrego sparse_output=False
# explícitamente porque HistGradientBoostingClassifier (uno de los seis
# modelos de este LAB) necesita entrada densa. En este dataset cat_cols
# queda vacío igual que en LAB02 (las categorías ya vienen codificadas
# como números), así que la rama cat_pipe no se activa en la práctica,
# pero la dejo correcta para no romper nada si algún día cambio de fuente.
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.20, random_state=42, stratify=y)
print("num_cols:", len(num_cols), "cat_cols:", len(cat_cols))
print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

Confirmo que `Xtr`/`Xte` dan la misma forma que en LAB02 (2520/630 filas, 14 columnas, `cat_cols=0`) — es la misma partición, no una nueva. A partir de acá empieza lo nuevo de este LAB.

### Paso 2 — Catálogo de seis configuraciones

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "logistic": Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]),
    "svm_c1": Pipeline([("prep", preprocess), ("model", SVC(C=1, probability=True, random_state=42))]),
    "svm_c10": Pipeline([("prep", preprocess), ("model", SVC(C=10, probability=True, random_state=42))]),
    "rf_100": Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))]),
    "rf_300": Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))]),
    "boost": Pipeline([("prep", preprocess), ("model", HistGradientBoostingClassifier(max_depth=6, random_state=42))]),
}
list(models)

Elegí estas seis para poder comparar tres familias distintas de modelo, no solo hiperparámetros del mismo: una lineal (logistic, mi nuevo baseline "barato"), dos SVM con `C` distinto (para ver si repetir la búsqueda de C10 sobre este catálogo más grande cambia algo respecto a LAB02) y dos ensambles (Random Forest con 100 y 300 árboles, y un HistGradientBoosting). `rf_100` vs `rf_300` es a propósito: si 300 árboles no mejoran el F1 de forma notoria respecto a 100, eso ya es evidencia en contra de "más árboles siempre ayuda" antes de llegar siquiera al análisis de Pareto.

### Paso 3 — Medir tres veces (mediana) y guardar cada modelo

In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
from sklearn.metrics import f1_score, recall_score

# Mismo patrón de detección de raíz que uso para reports/ en todo el
# proyecto, para que "reports/models" caiga siempre en la raíz del repo
# y no en notebooks/reports/models por el cwd de Jupyter.
project_root = Path.cwd()
if not (project_root / "requirements.txt").exists() and (project_root.parent / "requirements.txt").exists():
    project_root = project_root.parent
models_dir = project_root / "reports" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

rows = []
for name, model in models.items():
    fit_times = []
    for repetition in range(3):
        start = perf_counter()
        model.fit(Xtr, ytr)
        fit_times.append(perf_counter() - start)
    start = perf_counter()
    pred = model.predict(Xte)
    predict_ms = (perf_counter() - start) * 1000
    model_path = models_dir / f"{name}.joblib"
    joblib.dump(model, model_path)
    rows.append({
        "model": name,
        "f1_macro": f1_score(yte, pred, average="macro"),
        "recall_macro": recall_score(yte, pred, average="macro"),
        "recall_churn": recall_score(yte, pred, pos_label=1),
        "fit_median_s": np.median(fit_times),
        "predict_ms": predict_ms,
        "size_kb": model_path.stat().st_size / 1024,
    })

results = pd.DataFrame(rows)
results.sort_values("f1_macro", ascending=False)

Corrí esto con `random_state=42` en todos los modelos que lo aceptan, así que a ti te debería dar prácticamente lo mismo (los tiempos van a variar según tu máquina, eso es normal y lo trato como tal en el paso 9). A mí me dio, ordenado por F1 macro descendente: **boost 0.943**, **rf_100 0.891**, **svm_c10 0.886**, **rf_300 0.884**, **svm_c1 0.806**, **logistic 0.750**.

Dos cosas me llaman la atención de esta tabla antes de llegar a Pareto. Primero, `rf_300` (300 árboles) da **peor** F1 macro que `rf_100` (0.884 contra 0.891) y tarda casi tres veces más en ajustar (0.600 s contra 0.216 s) — la evidencia contra "más árboles siempre ayuda" que anticipé arriba se confirma. Segundo, `boost` no solo tiene el F1 más alto de los seis, sino que su mediana de ajuste (0.144 s) es más rápida que las dos SVM y que ambos Random Forest: no hay ningún trade-off que justifique elegir otro modelo que no sea `boost` o el `logistic` más barato, lo cual ya adelanta cómo se ve la frontera de Pareto en el paso 6.

### Paso 4 — PCA sobre el mismo split

In [ ]:
from sklearn.decomposition import PCA

# El bloque de X ya es totalmente numérico en este dataset (cat_cols
# quedó vacío desde LAB02), así que aplico PCA directo sobre las columnas
# numéricas sin necesidad de aislar un ColumnTransformer para eso. Si el
# dataset tuviera categorías sin codificar, aplicaría PCA solo al bloque
# numérico (o TruncatedSVD si viniera disperso tras un OneHot), nunca
# sobre columnas categóricas ya que la varianza ahí no tiene la misma
# interpretación geométrica.
Xtr_num = Xtr.select_dtypes(include="number")
Xte_num = Xte.select_dtypes(include="number")

svm_pca = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=.95, random_state=42)),
    ("model", SVC(C=1, probability=True, random_state=42)),
])
svm_pca.fit(Xtr_num, ytr)
n_components = svm_pca.named_steps["pca"].n_components_
f1_pca = f1_score(yte, svm_pca.predict(Xte_num), average="macro")
print("Componentes retenidos:", n_components, "de", Xtr_num.shape[1])
print("F1 macro con PCA (95% varianza):", round(f1_pca, 4))
print("F1 macro sin PCA (svm_c1 del catálogo):", round(results.loc[results.model=='svm_c1','f1_macro'].iloc[0], 4))

PCA con 95% de varianza retenida se queda con **8 de las 14 columnas** — una reducción real de dimensionalidad, no cosmética. El F1 macro baja de 0.806 (SVM C=1 sin PCA) a 0.799 con PCA: una diferencia de apenas 0.007, prácticamente dentro del ruido de un solo split. Mi lectura: para este dataset, casi todo lo que le sirve al SVM ya está concentrado en ese subespacio de 8 componentes, así que PCA aquí es sobre todo una herramienta de eficiencia/visualización (más útil todavía para el t-SNE del paso siguiente) más que una que yo necesitaría para ganar desempeño.

### Paso 5 — Dos mapas t-SNE con semillas distintas

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

X_num = X.select_dtypes(include="number").fillna(X.select_dtypes(include="number").median())
sample = X_num.sample(min(1000, len(X_num)), random_state=42)
y_sample = y.loc[sample.index]
scaled = StandardScaler().fit_transform(sample)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, seed in zip(axes, [42, 7]):
    emb = TSNE(n_components=2, perplexity=30, random_state=seed, init="pca", learning_rate="auto").fit_transform(scaled)
    ax.scatter(emb[:, 0], emb[:, 1], c=pd.Categorical(y_sample).codes, s=12, cmap="viridis")
    ax.set_title(f"t-SNE seed={seed}")
plt.tight_layout()
plt.savefig(project_root / "reports" / "tsne_two_seeds.png", dpi=160)
plt.show()

Uso una muestra de 1000 filas (no las 3150 completas) porque t-SNE escala mal en tiempo con el tamaño de la muestra, y para ilustrar la forma general de la nube esto alcanza. Entre las dos semillas, lo que **permanece** es la estructura gruesa: el mismo grupo compacto de puntos amarillos (clase `Churn=1`) abajo a la izquierda aparece en ambas corridas, y el resto de los clusters violeta también se mantiene en posiciones relativas similares. Lo que **cambia** es la orientación/rotación exacta del embedding y algunos detalles finos de qué tan separados quedan los sub-clusters intermedios — típico de t-SNE, que no tiene una orientación única y es sensible a la inicialización aleatoria (uso `init="pca"` justamente para reducir, no eliminar, esa variabilidad).

Ninguna de las dos imágenes demuestra por sí sola que un clasificador vaya a funcionar bien: la separación visual que se ve acá (el cluster amarillo compacto) es sugerente, pero es una proyección no lineal de 14 a 2 dimensiones pensada para visualizar, no una prueba de separabilidad ni evidencia de causalidad entre las variables y el churn. El SVM y los ensambles del paso 2 ya me dan la evaluación real; esto es solo una forma de mirar los datos, no un sustituto de esa evaluación.

### Paso 6 — Frontera de Pareto

In [ ]:
from inf8239_u01.green import pareto_flags

results["is_pareto"] = pareto_flags(results)
results.to_csv(project_root / "reports" / "green_ai_results.csv", index=False)
results.sort_values(["is_pareto", "f1_macro"], ascending=[False, False])

Traigo `pareto_flags` desde `src/inf8239_u01/green.py` en vez de definirla acá mismo, porque `tests/test_green.py` la importa igual (`from inf8239_u01.green import pareto_flags`) y quiero una sola fuente de verdad para esa lógica, no una copia en el notebook y otra en el módulo que se puedan desincronizar.

Con las seis configuraciones, la frontera queda formada por **`boost`** y **`logistic`** — exactamente los dos modelos que ya señalé como candidatos en el paso 3. Los otros cuatro (`rf_100`, `svm_c10`, `rf_300`, `svm_c1`) quedan dominados porque `boost` los supera en F1 macro y además tarda menos en ajustar que todos ellos.

### Paso 7 — Gráfico de la frontera de Pareto

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(results.fit_median_s, results.f1_macro,
           c=results.is_pareto.map({True: "#0f7c7b", False: "#94a3b8"}))
for _, r in results.iterrows():
    ax.annotate(r.model, (r.fit_median_s, r.f1_macro))
ax.set(xlabel="Mediana de ajuste (s)", ylabel="F1 macro", title="Rendimiento y costo")
plt.tight_layout()
plt.savefig(project_root / "reports" / "pareto.png", dpi=160)
plt.show()

### Análisis de la decisión Pareto (300–500 palabras)

Con las seis configuraciones evaluadas, la frontera de Pareto entre F1 macro y mediana de tiempo de ajuste queda formada por solo dos modelos: boost (HistGradientBoosting) y logistic (regresión logística). Los otros cuatro —rf_100, svm_c10, rf_300 y svm_c1— quedan estrictamente dominados: existe al menos otro modelo (boost) que logra un F1 macro más alto y un tiempo de ajuste más bajo al mismo tiempo, así que no tiene sentido elegirlos bajo ningún balance razonable entre desempeño y costo.

El F1 macro máximo lo obtiene boost, con 0.943, casi diez puntos por encima del segundo mejor no dominado por él (rf_100, 0.891). Lo que hace interesante a boost no es solo ese máximo, sino que lo logra con una mediana de ajuste de apenas 0.144 s, más rápida que las dos variantes de SVM y que ambos Random Forest — es decir, domina en las dos dimensiones a la vez a cuatro de los cinco competidores restantes.

La alternativa más barata en la frontera es logistic: 0.012 s de mediana de ajuste, un 91.7% menos tiempo que boost, y el modelo serializado más liviano con diferencia (4.1 KB contra 316.7 KB de boost). Pero esa economía tiene un costo real en la métrica que definí como prioritaria desde la ficha del dataset: el recall de la clase Churn=1. logistic detecta solo el 42% de los clientes que realmente se van (recall_churn=0.424), mientras que boost detecta el 91% (0.909). La diferencia absoluta de F1 macro entre ambos es de 0.193 puntos, y en términos del error más costoso del proyecto —el falso negativo de retención— logistic deja sin detectar más de la mitad de los clientes que se van, contra menos de una décima parte con boost.

Mi decisión es quedarme con boost, no con la opción más barata de la frontera. La razón es de contexto de negocio, no solo de números: 0.144 s de ajuste y 316 KB de modelo son costos triviales en términos absolutos para un proceso de scoring que corre, como mucho, una vez al día sobre unos pocos miles de clientes; el ahorro de 91.7% de tiempo que ofrece logistic no compensa perder más de la mitad del recall sobre la clase que me importa.

Limitaciones: estas medidas de tiempo y tamaño son proxies de costo computacional, no consumo energético ni huella de CO2 —no medí ninguna de las dos, y no las voy a presentar como si lo hubiera hecho—. Los tiempos además son relativos a esta máquina y a esta única corrida (Linux x86_64, scikit-learn 1.8.0, sin otros procesos compitiendo por CPU); en el hardware del alumno, o bajo carga distinta, la magnitud absoluta cambiaría aunque el orden relativo entre modelos probablemente se mantenga.


### Paso 9 — Registro del entorno

In [ ]:
import platform
import sklearn

print("Python", sys.version)
print("Sistema", platform.platform())
print("Procesador", platform.processor())
print("scikit-learn", sklearn.__version__)

Anoto esto porque el tiempo de ajuste que reporto en este notebook es **contextual**, no una medida absoluta de eficiencia energética: depende de esta máquina, de qué tan cargada esté de otros procesos en el momento de correr, y de la versión de scikit-learn (los algoritmos internos cambian entre versiones). No medí consumo energético ni CO2 en ningún momento de este LAB, y por eso tampoco lo presento en la conclusión ni en `reports/green_ai_results.csv` — si en el futuro quisiera medir eso de verdad, tendría que identificar explícitamente la herramienta (por ejemplo `codecarbon` o `pyJoules`), la región eléctrica de donde corre el cómputo y qué parte del pipeline cubre la medición (solo el `fit`, o también la carga de datos y el preprocesamiento), porque cada una de esas decisiones cambia el número final.

### Paso 10 — Checklist final

- [x] Mismo dataset y misma partición del Ejercicio 01 (LAB02): reconstruidos en el paso 1, sin cambiar `TARGET`, `DROP_COLUMNS` ni `random_state`.
- [x] Al menos seis configuraciones evaluadas: `logistic`, `svm_c1`, `svm_c10`, `rf_100`, `rf_300`, `boost`.
- [x] Incluye SVM, Random Forest y boosting: las tres familias están en el catálogo del paso 2.
- [x] PCA aplicado y justificado (paso 4): 8 de 14 componentes retienen 95% de varianza; comparado explícitamente contra el SVM sin PCA.
- [x] Dos mapas t-SNE con semillas distintas (paso 5), con la lectura de qué permanece/cambia y la advertencia de que no prueba causalidad.
- [x] Tres repeticiones de tiempo de ajuste y mediana reportada (paso 3), no un solo tiempo de una corrida.
- [x] Tamaño serializado (`.joblib`) y tiempo de inferencia medidos y guardados por modelo (paso 3).
- [x] CSV (`reports/green_ai_results.csv`), figuras (`reports/tsne_two_seeds.png`, `reports/pareto.png`), pruebas (`tests/test_green.py`) y README actualizados.
- [x] Frontera de Pareto calculada (`pareto_flags`, paso 6) y decisión cuantificada por escrito (paso 8), con el modelo elegido (`boost`) justificado frente a la alternativa más barata de la frontera (`logistic`).